# SmogSense: Phase 1 Baseline Ladder Showcase (M0–M3b) & City-Panel Target

**Objective:** Provide an interactive, Google Colab-ready portfolio showcase and verification evaluation of all five Phase 1 statistical & physical baselines (`M0`, `M1`, `M2`, `M3`, `M3b`) and the City-Panel Pseudo-Location Target (`compute_city_panel_target`).

### The Phase 1 Baseline Verification Ladder:
| Model | Name | Method Formulation | Role on Ladder |
|---|---|---|---|
| **M0** | Persistence | $y_{T+h} = \bar{y}_{T-24:T} + \epsilon_\tau$ | Naive benchmark; undefined at $N=0$ |
| **M1** | Raw CAMS | $y_{T+h} = \text{CAMS}(T+h) + \epsilon_\tau$ | Operational NWP baseline (deterministic at $N=0$) |
| **M2** | CAMS Bias-Corrected | $\hat{y} = \alpha + \beta \cdot \text{CAMS}$; $q_\tau = \hat{y} + \epsilon_\tau$ | **Primary reference benchmark** (H1/H2 hypothesis anchor) |
| **M3** | Seasonal Climatology | 19 quantiles from DOY $\pm 15$-day window | Seasonal diurnal benchmark |
| **M3b**| Linear Quantile Regression | 19 independent linear quantile regressors on $\{y_{\text{lag}}, \text{CAMS}, \text{BLH}, \text{Wind}\}$ | Linear machine learning quantile benchmark |

Every model operates strictly on information available at or before issuance reference time $T$ (`available_at_utc <= T`), emitting 19 monotonically rearranged quantiles $\tau \in \{0.05, 0.10, \dots, 0.95\}$ through `smogsense.models.distribution.QuantileFunction`.

In [ ]:
# 0. Environment Setup (Google Colab & Local Parity)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    print("Running on Google Colab. Cloning repository and installing dependencies...")
    !git clone --depth 1 https://github.com/haiderkashan/smog-sense.git /content/smogsense_repo
    %cd /content/smogsense_repo
    !pip install -q "numpy>=2.0" "pandas>=2.2" "scipy>=1.14" "scikit-learn>=1.5" "pyproj>=3.6" "pydantic>=2.7" "typer>=0.12" "matplotlib>=3.8" "pyarrow>=16.0" "structlog"
    if "/content/smogsense_repo/src" not in sys.path:
        sys.path.insert(0, "/content/smogsense_repo/src")
else:
    repo_root = Path.cwd().resolve()
    while repo_root.parent != repo_root and not (repo_root / "pyproject.toml").exists():
        repo_root = repo_root.parent
    src_path = str(repo_root / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from smogsense.evaluation.scoring import compute_crps, interval_score, pinball_loss
from smogsense.features.targets import compute_city_panel_target
from smogsense.models.baselines import (
    M0Persistence,
    M1Cams,
    M2CamsBiasCorrected,
    M3bLinearQuantileRegression,
    M3Climatology,
)

print("✓ SmogSense baseline engine and evaluation modules loaded.")

## 1. Smog Season Data Preparation

We assemble a 60-day historical training and evaluation dataset for Lahore during the winter smog season (October–December).
If historical Parquet partitions exist under `data/processed/backfill/`, we load them; otherwise, we construct a realistic 60-day Lahore dataset featuring regulatory reference monitor `1001` (US Embassy Lahore) alongside co-located low-cost sensors, capturing diurnal boundary-layer stagnation and high-pollution episodes.

In [ ]:
from datetime import UTC, datetime, timedelta

# Define 60-day smog season window for Lahore
start_dt = datetime(2026, 10, 1, 0, 0, tzinfo=UTC)
n_days = 60
timestamps = [start_dt + timedelta(hours=i) for i in range(n_days * 24)]

# Construct 4 stations: Station 1001 (reference), Station 1002 (co-located within 25m), Station 2001 (isolated low-cost), Station 3001 (low-cost)
stations_registry = pd.DataFrame(
    [
        {
            "location_id": 1001,
            "name": "US Embassy Lahore",
            "lat": 31.5501,
            "lon": 74.3302,
            "is_reference": True,
            "provider": "AirNow",
            "eligible": True,
        },
        {
            "location_id": 1002,
            "name": "Embassy Co-located LowCost",
            "lat": 31.5502,
            "lon": 74.3303,
            "is_reference": False,
            "provider": "PurpleAir",
            "eligible": True,
        },
        {
            "location_id": 2001,
            "name": "Gulberg Commercial",
            "lat": 31.5204,
            "lon": 74.3587,
            "is_reference": False,
            "provider": "Clarity",
            "eligible": True,
        },
        {
            "location_id": 3001,
            "name": "Johar Town Residential",
            "lat": 31.4700,
            "lon": 74.2700,
            "is_reference": False,
            "provider": "PurpleAir",
            "eligible": True,
        },
    ]
)

# Generate synthetic realistic hourly smog-season PM2.5, CAMS forecasts, BLH, and wind speed
rng = np.random.default_rng(42)
records = []
for ts in timestamps:
    doy = ts.timetuple().tm_yday
    hour = ts.hour
    # Smog season builds up in Nov-Dec (DOY 305-340)
    seasonal_trend = 60.0 + 90.0 * np.exp(-((doy - 315) ** 2) / 250.0)
    # Diurnal peak during night stagnation
    diurnal = 35.0 * np.cos((hour - 4) * 2 * np.pi / 24.0)
    blh = max(100.0, 750.0 - diurnal * 12.0 + float(rng.normal(0, 40)))
    wind = max(0.5, 3.2 - diurnal * 0.05 + float(rng.normal(0, 0.4)))
    # CAMS global forecast typically under-predicts smog peaks (systematic slope ~0.65, intercept ~15)
    cams_pm25 = max(10.0, 0.65 * (seasonal_trend + diurnal) + 15.0 + float(rng.normal(0, 12.0)))

    for _, stn in stations_registry.iterrows():
        lid = stn["location_id"]
        noise = float(rng.normal(0, 6.0 if stn["is_reference"] else 14.0))
        pm_val = max(5.0, seasonal_trend + diurnal + noise)
        records.append(
            {
                "location_id": lid,
                "ts_utc": ts,
                "available_at_utc": ts + timedelta(hours=4),
                "pm25_ugm3": pm_val,
                "cams_pm25": cams_pm25,
                "blh_m": blh,
                "wind_speed": wind,
                "imputed": False,
            }
        )

df_observations = pd.DataFrame(records)
print(
    f"✓ Generated {len(df_observations)} hourly observations across {len(stations_registry)} stations for {n_days} days."
)
df_observations[["ts_utc", "location_id", "pm25_ugm3", "cams_pm25", "blh_m", "wind_speed"]].head(6)

## 2. City-Panel Pseudo-Location & 50m Co-location Hierarchy (Task P1-21)

Under ADR-018 and `docs/ml-architecture.md §1`, the city target is a pseudo-location defined as the mean of group-collapsed station 24-hour block means over a frozen panel (requiring $\ge 3$ valid groups and $\ge 18/24$ valid non-imputed hours per station).
When stations are co-located within 50 meters, the hierarchy **strictly prefers regulatory reference monitors** (`is_reference=True`) over low-cost sensors.

In [ ]:
# Test city panel target on a sample 24-hour block
sample_day = df_observations[df_observations["ts_utc"] < start_dt + timedelta(hours=24)]

panel_result = compute_city_panel_target(
    sample_day,
    registry_df=stations_registry,
    min_valid_stations=3,
    min_valid_hours=18,
    colocation_radius_m=50.0,
)

print("=== City Panel Target Aggregation Result ===")
print(f"Target PM2.5 (y_city):        {panel_result.y_city:.2f} µg/m³")
print(f"Panel Valid:                  {panel_result.valid}")
print(f"Valid Station Groups:         {panel_result.n_valid_stations}")
print(f"Raw Stations Reporting:       {panel_result.n_raw_valid_stations}")
print(f"Reference Monitors Count:     {panel_result.n_reference}")
print(f"Low-Cost Groups Count:        {panel_result.n_low_cost}")
print(f"Overall Truth Basis:          {panel_result.truth_basis}")
print(f"Collapsed Group Means:        {panel_result.group_means}")

## 3. Fitting All 5 Phase 1 Baselines (M0, M1, M2, M3, M3b)

We partition our dataset into:
- **Historical calibration window** (first 40 days, available at reference time $T$)
- **Out-of-sample evaluation window** (last 20 days, scored prospectively)

We fit `M0`, `M1`, `M2` (both linear and log1p mode), `M3` (circular DOY), and `M3b` (19 linear quantile regressors with monotone rearrangement).

In [ ]:
# Reference forecast time T at day 40
t_forecast = start_dt + timedelta(days=40)
history_df = df_observations[df_observations["available_at_utc"] <= t_forecast].copy()
eval_df = df_observations[df_observations["ts_utc"] >= t_forecast].copy()

# Station 1001 historical pairs
stn_history = history_df[history_df["location_id"] == 1001].copy()
recent_24h = stn_history[stn_history["ts_utc"] >= t_forecast - timedelta(hours=24)][
    "pm25_ugm3"
].to_numpy()

# 1. Baseline M0: Persistence
resids_m0 = np.quantile(
    stn_history["pm25_ugm3"].to_numpy() - np.roll(stn_history["pm25_ugm3"].to_numpy(), 24)[24:],
    np.linspace(0.05, 0.95, 19),
)
model_m0 = M0Persistence(residual_quantiles=resids_m0)
pred_m0 = model_m0.predict(recent_24h)

# 2. Baseline M1: Raw CAMS
resids_m1 = np.quantile(
    stn_history["pm25_ugm3"].to_numpy() - stn_history["cams_pm25"].to_numpy(),
    np.linspace(0.05, 0.95, 19),
)
model_m1 = M1Cams(residual_quantiles=resids_m1)
current_cams_fc = float(eval_df[eval_df["location_id"] == 1001]["cams_pm25"].iloc[0])
pred_m1 = model_m1.predict(current_cams_fc)

# 3. Baseline M2: CAMS Bias-Corrected (Linear and Log-space modes)
model_m2_lin = M2CamsBiasCorrected(
    history_df=stn_history, min_samples=7, window_days=30, log_space=False
)
model_m2_log = M2CamsBiasCorrected(
    history_df=stn_history, min_samples=7, window_days=30, log_space=True
)
pred_m2 = model_m2_lin.predict(current_cams_fc)

print(
    f"M2 (Linear) Fit: Slope = {model_m2_lin.slope_:.3f}, Intercept = {model_m2_lin.intercept_:.2f} µg/m³ (N={model_m2_lin.n_samples_})"
)
print(
    f"M2 (Log1p)  Fit: Slope = {model_m2_log.slope_:.3f}, Intercept = {model_m2_log.intercept_:.2f}"
)

# 4. Baseline M3: Climatology (Circular Day-of-Year)
model_m3 = M3Climatology(history_df=stn_history, window_half_days=15, min_window_samples=10)
pred_m3 = model_m3.predict(target_date=t_forecast, horizon_h=24)

# 5. Baseline M3b: Linear Quantile Regression (19 levels, L1 regularisation, monotone rearrangement)
stn_history["lag_24h_pm25"] = stn_history["pm25_ugm3"].shift(24).bfill()
features = ["lag_24h_pm25", "cams_pm25", "blh_m", "wind_speed"]
model_m3b = M3bLinearQuantileRegression(feature_cols=features, alpha=1e-3, min_samples=20)
model_m3b.fit(stn_history[features], y=stn_history["pm25_ugm3"].to_numpy())
eval_features = stn_history[features].iloc[[-1]]
pred_m3b = model_m3b.predict(eval_features)

print("✓ All 5 baselines successfully fitted on available history A.")

## 4. Visualizations: Systematic Bias Removal & 72-Hour Probabilistic Fan Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Raw CAMS vs. Observations with 1:1 line vs M2 Bias-Correction fit
ax1 = axes[0]
c_pts = stn_history["cams_pm25"].to_numpy()
y_pts = stn_history["pm25_ugm3"].to_numpy()
ax1.scatter(c_pts, y_pts, alpha=0.35, color="steelblue", label="Observed Pairs")
grid_x = np.linspace(10, max(c_pts.max(), y_pts.max()), 100)
ax1.plot(grid_x, grid_x, "k--", alpha=0.7, label="1:1 Identity Line")
ax1.plot(
    grid_x,
    model_m2_lin.intercept_ + model_m2_lin.slope_ * grid_x,
    color="firebrick",
    lw=2.5,
    label=f"M2 Fit (slope={model_m2_lin.slope_:.2f}, int={model_m2_lin.intercept_:.1f})",
)
ax1.set_title("Plot 1: CAMS Systematic Bias vs. M2 Bias Correction", fontsize=12, fontweight="bold")
ax1.set_xlabel("Raw CAMS Forecast PM2.5 (µg/m³)")
ax1.set_ylabel("Observed Ground PM2.5 (µg/m³)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Probabilistic Quantile Profiles Across Baselines
ax2 = axes[1]
tau_levels = np.linspace(0.05, 0.95, 19)
ax2.plot(tau_levels, pred_m0.q, label="M0 Persistence", marker="o", lw=1.8)
ax2.plot(tau_levels, pred_m1.q, label="M1 Raw CAMS", marker="s", lw=1.8)
ax2.plot(tau_levels, pred_m2.q, label="M2 CAMS-BC (Ref)", marker="^", lw=2.5, color="firebrick")
ax2.plot(tau_levels, pred_m3.q, label="M3 Climatology", marker="d", lw=1.8)
ax2.plot(tau_levels, pred_m3b.q, label="M3b Linear QR", marker="v", lw=2.2, color="darkgreen")
ax2.set_title(
    "Plot 2: Probabilistic 19-Quantile Curves (tau = 0.05...0.95)", fontsize=12, fontweight="bold"
)
ax2.set_xlabel("Quantile Level (tau)")
ax2.set_ylabel("Predicted PM2.5 (µg/m³)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Prospective Verification Leaderboard

We evaluate Continuous Ranked Probability Score (**CRPS**), Continuous Ranked Probability Skill Score (**CRPSS**) relative to primary reference `M2`, **Pinball Loss** at $\tau \in \{0.50, 0.90\}$, and 80% Interval Score (**IS_80**) over the prospective evaluation window.

In [ ]:
# Score each baseline over the out-of-sample evaluation period
eval_stn = eval_df[eval_df["location_id"] == 1001].copy().reset_index(drop=True)
eval_stn["lag_24h_pm25"] = eval_stn["pm25_ugm3"].shift(24).bfill()

metrics_records = []
models_dict = {
    "M0 Persistence": lambda row: model_m0.predict(recent_24h),
    "M1 Raw CAMS": lambda row: model_m1.predict(row["cams_pm25"]),
    "M2 CAMS-BC": lambda row: model_m2_lin.predict(row["cams_pm25"]),
    "M3 Climatology": lambda row: model_m3.predict(target_date=row["ts_utc"], horizon_h=24),
    "M3b Linear QR": lambda row: model_m3b.predict(row[features].to_frame().T),
}

crps_by_model = {}
for name, pred_fn in models_dict.items():
    crps_list = []
    pb50_list = []
    pb90_list = []
    is80_list = []
    for _, row in eval_stn.iloc[:72].iterrows():
        y_true = row["pm25_ugm3"]
        q_func = pred_fn(row)
        crps_list.append(compute_crps(y_true, q_func.q))
        pb50_list.append(pinball_loss(y_true, q_func.q50, 0.50))
        pb90_list.append(pinball_loss(y_true, q_func.q90, 0.90))
        is80_list.append(interval_score(y_true, q_func.q10, q_func.q90, alpha=0.20))

    mean_crps = float(np.mean(crps_list))
    crps_by_model[name] = mean_crps
    metrics_records.append(
        {
            "Model": name,
            "CRPS (µg/m³)": round(mean_crps, 2),
            "Pinball q50": round(float(np.mean(pb50_list)), 2),
            "Pinball q90": round(float(np.mean(pb90_list)), 2),
            "Interval Score 80pct": round(float(np.mean(is80_list)), 2),
        }
    )

leaderboard_df = pd.DataFrame(metrics_records)
m2_ref_crps = crps_by_model["M2 CAMS-BC"]
leaderboard_df["CRPSS vs M2"] = leaderboard_df["CRPS (µg/m³)"].apply(
    lambda v: round(1.0 - (v / m2_ref_crps), 3)
)

print("=== SmogSense Phase 1 Baseline Verification Leaderboard ===")
display(leaderboard_df)
print("✓ All Phase 1 baselines verified and benchmarked against M2 reference anchor.")